# Training Pipeline
[run_training_dpo_pipeline.ipynb](https://github.com/shibing624/MedicalGPT/blob/main/notebooks/run_training_dpo_pipeline.ipynb)    | [Open In Colab](https://colab.research.google.com/github/zhifeiluo7/MedicalGPT_Project/blob/main/notebooks/run_training_dpo_pipeline.ipynb)

# 虚拟环境预处理

配置一个干净的虚拟环境，避免Colab的煞笔环境冲突

In [1]:
!pip install -q condacolab
import condacolab
condacolab.install()

✨🍰✨ Everything looks OK!


In [2]:
!conda create -n medgpt python=3.10 -y
!source activate medgpt

Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/medgpt

  added / updated specs:
    - python=3.10


The following NEW packages will be INSTALLED:

  _openmp_mutex      conda-forge/linux-64::_openmp_mutex-4.5-20_gnu 
  bzip2              conda-forge/linux-64::bzip2-1.0.8-hda65f42_9 
  ca-certificates    conda-forge/noarch::ca-certificates-2026.5.20-hbd8a1cb_0 
  ld_impl_linux-64   conda-forge/linux-64::ld_impl_linux-64-2.45.1-default_hbd61a6d_102 
  libexpat           conda-forge/linux-64::libexpat-2.8.1-hecca717_0 
  libffi             conda-forge/linux-64::libffi-3.5.2-h3435931_0 
  libgcc             conda-forge/linux-64::libgcc-15.2.0-he0feb66_19 
  libgcc-ng          conda-forge/linux-64::libgcc-ng-15.2.

In [3]:
!conda install pip -y
!pip install transformers peft torch torchvision torchaudio

Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - \ done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

  Using cached transformers-5.10.2-py3-none-any.whl.metadata (33 kB)
  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.10.2-py3-none-any.whl (11.0 MB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
Using cached peft-0.19.1-py3-none-any.whl (680 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [peft]


# Stage 1: Continue Pretraining

第一阶段：PT(Continue PreTraining)增量预训练，在海量领域文本数据上二次预训练GPT模型，以适配领域数据分布

注意：
1. 此阶段是可选的，如果你没有海量领域文本，可以跳过此阶段，直接进行SFT阶段的有监督微调
2. 我实验发现：做领域知识注入，SFT比PT更高效，也可以跳过PT阶段

| Stage 1: Continue Pretraining   |  [pretraining.py](https://github.com/shibing624/MedicalGPT/blob/main/training/pretraining.py) | [run_pt.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_pt.sh)    |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B
2. 数据集：PT阶段使用的是中文天龙八部小说部分文本和英文书籍部分文本，位于`data/pretrain`文件夹

## 配置运行环境

本地执行可注释以下配置环境的命令，colab执行要打开注释，用于配置环境

colab建议使用T4 GPU训练，设置方式：`代码执行程序 -> 更改运行时类型 -> 运行时类型：Python3，硬件加速器：GPU，GPU类型：T4 -> 保存`

步骤：
1. 下载最新代码到本地
2. 安装依赖包

依赖包如下，保证最新版本：

```
loguru
transformers
sentencepiece
datasets
tensorboard
tqdm
peft
trl
torchao> 0.16.0
```

In [7]:
!rm -rf MedicalGPT_Project
!git clone --depth 1 https://github.com/zhifeiluo7/MedicalGPT_Project.git
%cd MedicalGPT_Project
%ls
!pip install --upgrade "torchao>0.16.0"
!pip install -r requirements.txt

Cloning into 'MedicalGPT_Project'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 136 (delta 27), reused 128 (delta 26), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 7.66 MiB | 15.66 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/MedicalGPT_Project
CITATION.cff     data/       docs/    notebooks/    requirements.txt  tests/
_config.yml      demo/       LICENSE  README_EN.md  role_play_data/   tools/
CONTRIBUTING.md  DISCLAIMER  llm/     README.md     scripts/          training/
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 30.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 85.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 167.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 175.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 169.8 MB/s  0:00:00
   ━━━

### Stage1 PT

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

**以下参数可以根据GPU实际情况修改，当前参数是根据Colab的T4单卡GPU（16GB显存）配置的**

In [25]:
%ls ./data/pretrain/

en_article_tail500.txt  fever.txt  tianlongbabu.txt


In [26]:
!python training/pretraining.py \
    --model_name_or_path Qwen/Qwen2.5-0.5B \
    --train_file_dir ./data/pretrain \
    --validation_file_dir ./data/pretrain \
    --per_device_train_batch_size 3 \
    --per_device_eval_batch_size 3 \
    --do_train \
    --do_eval \
    --use_peft True \
    --seed 42 \
    --bf16 \
    --max_train_samples 20000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-4 \
    --warmup_ratio 0.05 \
    --weight_decay 0.01 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 50 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 1 \
    --preprocessing_num_workers 1 \
    --block_size 128 \
    --output_dir outputs-pt-v1 \
    --overwrite_output_dir \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --gradient_checkpointing True

W0609 03:34:55.123000 10302 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0609 03:34:55.148000 10302 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-06-09 03:34:55.616 | INFO     | __main__:main:364 - Model args: ModelArguments(model_name_or_path='Qwen/Qwen2.5-0.5B', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=False, cache_dir=None, model_revision='main', hf_hub_token=None, use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='au

In [27]:
%ls -lh outputs-pt-v1

total 28M
-rw-r--r-- 1 root root 1.1K Jun  9 03:45 adapter_config.json
-rw-r--r-- 1 root root  17M Jun  9 03:45 adapter_model.safetensors
-rw-r--r-- 1 root root  470 Jun  9 03:45 all_results.json
-rw-r--r-- 1 root root 2.4K Jun  9 03:45 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Jun  9 03:44 checkpoint-750/
drwxr-xr-x 2 root root 4.0K Jun  9 03:45 checkpoint-800/
drwxr-xr-x 2 root root 4.0K Jun  9 03:45 checkpoint-845/
-rw-r--r-- 1 root root  262 Jun  9 03:45 eval_results.json
-rw-r--r-- 1 root root 5.1K Jun  9 03:45 README.md
drwxr-xr-x 4 root root 4.0K Jun  9 03:35 runs/
-rw-r--r-- 1 root root  723 Jun  9 03:45 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  9 03:45 tokenizer.json
-rw-r--r-- 1 root root  21K Jun  9 03:45 trainer_state.json
-rw-r--r-- 1 root root  228 Jun  9 03:45 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [36]:
!wget -O tools/merge_peft_adapter.py https://raw.githubusercontent.com/zhifeiluo7/MedicalGPT_Project/main/tools/merge_peft_adapter.py

--2026-06-09 03:53:09--  https://raw.githubusercontent.com/zhifeiluo7/MedicalGPT_Project/main/tools/merge_peft_adapter.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6250 (6.1K) [text/plain]
Saving to: ‘tools/merge_peft_adapter.py’

tools/merge_peft_ad 100%[===================>]   6.10K  --.-KB/s    in 0s      

2026-06-09 03:53:09 (69.4 MB/s) - ‘tools/merge_peft_adapter.py’ saved [6250/6250]



In [37]:
!python tools/merge_peft_adapter.py \
    --base_model Qwen/Qwen2.5-0.5B --lora_model outputs-pt-v1 --output_dir merged-pt/

W0609 03:53:18.774000 14995 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0609 03:53:18.800000 14995 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Namespace(base_model='Qwen/Qwen2.5-0.5B', tokenizer_path=None, lora_model='outputs-pt-v1', resize_emb=False, output_dir='merged-pt/', hf_hub_model_id='', hf_hub_token=None)
Base model: Qwen/Qwen2.5-0.5B
LoRA model: outputs-pt-v1
Loading LoRA for causal language model (archs=['Qwen2ForCausalLM'])
Loading weights: 100% 290/290 [00:00<00:00, 858.20it/s]
Merging with merge_and_unload...
Saving to Hugging Face forma

In [38]:
%ls -lh merged-pt/

total 954M
-rw-r--r-- 1 root root 2.4K Jun  9 03:53 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Jun  9 03:53 config.json
-rw-r--r-- 1 root root  139 Jun  9 03:53 generation_config.json
-rw-r--r-- 1 root root 943M Jun  9 03:53 model.safetensors
-rw-r--r-- 1 root root  697 Jun  9 03:53 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  9 03:53 tokenizer.json


In [39]:
%cat merged-pt/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage1 增量预训练完成。

# Stage 2: Supervised FineTuning

第二阶段：SFT(Supervised Fine-tuning)有监督微调，构造指令微调数据集，在预训练模型基础上做指令精调，以对齐指令意图，并注入领域知识

| Stage 2: Supervised Fine-tuning | [supervised_finetuning.py](https://github.com/shibing624/MedicalGPT/blob/main/training/supervised_finetuning.py) | [run_sft.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_sft.sh)  |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是Qwen/Qwen2.5-0.5B 或者 Stage1得到的预训练模型
2. 数据集：SFT阶段使用的是使用的是Belle的1千条抽样数据，位于`data/finetune`文件夹

## Stage2 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

In [40]:
%ls ./data/sft

glaive_toolcall_zh_demo.jsonl  sharegpt_zh_1K_format.jsonl
medical_sft_1K_format.jsonl


In [41]:
!python training/supervised_finetuning.py \
    --model_name_or_path merged-pt \
    --train_file_dir ./data/sft \
    --validation_file_dir ./data/sft \
    --per_device_train_batch_size 4 \
    --per_device_eval_batch_size 4 \
    --do_train \
    --do_eval \
    --use_peft True \
    --bf16 \
    --max_train_samples 1000 \
    --max_eval_samples 10 \
    --num_train_epochs 1 \
    --learning_rate 2e-5 \
    --warmup_ratio 0.05 \
    --weight_decay 0.05 \
    --logging_strategy steps \
    --logging_steps 10 \
    --eval_steps 50 \
    --eval_strategy steps \
    --save_steps 500 \
    --save_strategy steps \
    --save_total_limit 3 \
    --gradient_accumulation_steps 1 \
    --preprocessing_num_workers 1 \
    --output_dir outputs-sft-v1 \
    --ddp_timeout 30000 \
    --logging_first_step True \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --device_map auto \
    --report_to tensorboard \
    --ddp_find_unused_parameters False \
    --gradient_checkpointing True

W0609 03:54:07.175000 15219 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0609 03:54:07.201000 15219 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-06-09 03:54:07.624 | WARNING  | __main__:__post_init__:191 - You may set max_train_samples = -1 to run all samples in production.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
2026-06-09 03:54:07.852 | INFO     | __main__:main:352 - Model args: ModelArguments(model_name_or_path='merged-pt', load_in_8bit=False, load_in_4bit=False, tokenizer_name_or_path=N

In [42]:
%ls -lh outputs-sft-v1

total 28M
-rw-r--r-- 1 root root 1.1K Jun  9 04:07 adapter_config.json
-rw-r--r-- 1 root root  17M Jun  9 04:07 adapter_model.safetensors
-rw-r--r-- 1 root root  428 Jun  9 04:07 all_results.json
-rw-r--r-- 1 root root 2.4K Jun  9 04:07 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Jun  9 04:07 checkpoint-250/
-rw-r--r-- 1 root root  220 Jun  9 04:07 eval_results.json
-rw-r--r-- 1 root root 5.1K Jun  9 04:07 README.md
drwxr-xr-x 3 root root 4.0K Jun  9 03:54 runs/
-rw-r--r-- 1 root root  733 Jun  9 04:07 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  9 04:07 tokenizer.json
-rw-r--r-- 1 root root 6.3K Jun  9 04:07 trainer_state.json
-rw-r--r-- 1 root root  228 Jun  9 04:07 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [43]:
!python tools/merge_peft_adapter.py \
    --base_model merged-pt --lora_model outputs-sft-v1 --output_dir ./merged-sft

W0609 04:08:14.106000 18880 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0609 04:08:14.132000 18880 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Namespace(base_model='merged-pt', tokenizer_path=None, lora_model='outputs-sft-v1', resize_emb=False, output_dir='./merged-sft', hf_hub_model_id='', hf_hub_token=None)
Base model: merged-pt
LoRA model: outputs-sft-v1
Loading LoRA for causal language model (archs=['Qwen2ForCausalLM'])
Loading weights: 100% 290/290 [00:00<00:00, 793.15it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writing

In [44]:
%ls -lh merged-sft/

total 954M
-rw-r--r-- 1 root root 2.4K Jun  9 03:53 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Jun  9 04:08 config.json
-rw-r--r-- 1 root root  139 Jun  9 03:53 generation_config.json
-rw-r--r-- 1 root root 943M Jun  9 04:08 model.safetensors
-rw-r--r-- 1 root root  697 Jun  9 03:53 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  9 03:53 tokenizer.json


In [45]:
%cat merged-sft/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage2 SFT训练完成。

# Stage 3: DPO(Direct Preference Optimization)

第三阶段：DPO(Direct Preference Optimization)直接偏好优化，DPO通过直接优化语言模型来实现对其行为的精确控制，而无需使用复杂的强化学习，也可以有效学习到人类偏好，DPO相较于RLHF更容易实现且易于训练，效果更好

| Stage 3: Direct Preference Optimization        |  [dpo_training.py](https://github.com/shibing624/MedicalGPT/blob/main/training/dpo_training.py) | [run_dpo.sh](https://github.com/shibing624/MedicalGPT/blob/main/scripts/run_dpo.sh)    |

#### 说明：
以下 notebook/colab 代码为了快速验证训练代码可用，我们使用了小size的生成模型和小样本数据集，实际使用时，需要使用更大的模型和数据集，以获得更好的效果。

1. 生成模型：使用的是`Qwen/Qwen2.5-0.5B` 或者 Stage2得到的SFT模型
2. 数据集：DPO阶段使用的是医疗reward数据，抽样了500条，位于`data/reward`文件夹

## Stage3 咱们开始吧

训练步骤如下：

1. 确认训练集
2. 执行训练脚本

训练脚本的执行逻辑如下：
1. 导入依赖包
2. 设置参数
3. 定义各函数并加载训练集
4. 加载模型和tokenizer
5. 开始训练并评估
6. 查看训练结果

In [46]:
%ls ./data/reward/

dpo_zh_500.jsonl  toolcall_dpo_zh_demo.jsonl


In [47]:
!python training/dpo_training.py \
    --model_name_or_path ./merged-sft \
    --template_name qwen \
    --train_file_dir ./data/reward \
    --validation_file_dir ./data/reward \
    --per_device_train_batch_size 3 \
    --per_device_eval_batch_size 1 \
    --do_train \
    --do_eval \
    --use_peft True \
    --max_train_samples 1000 \
    --max_eval_samples 500 \
    --max_steps 100 \
    --eval_steps 10 \
    --save_steps 50 \
    --max_source_length 256 \
    --max_target_length 256 \
    --output_dir outputs-dpo-v1 \
    --target_modules all \
    --lora_rank 8 \
    --lora_alpha 16 \
    --lora_dropout 0.05 \
    --torch_dtype bfloat16 \
    --bf16 True \
    --fp16 False \
    --device_map auto \
    --report_to tensorboard \
    --remove_unused_columns False \
    --gradient_checkpointing True \
    --cache_dir ./cache

W0609 04:09:16.585000 19159 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0609 04:09:16.620000 19159 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
2026-06-09 04:09:17.295 | INFO     | __main__:main:208 - Parse args: ScriptArguments(model_name_or_path='./merged-sft', tokenizer_name_or_path=None, load_in_8bit=False, load_in_4bit=False, cache_dir='./cache', use_fast_tokenizer=False, torch_dtype='bfloat16', device_map='auto', trust_remote_code=True, dataset_name=None, dataset_config_name=None, train_file_dir='./data/reward', validation_file_dir='./data/reward

In [48]:
%ls -lh outputs-dpo-v1

total 28M
-rw-r--r-- 1 root root 1.1K Jun  9 04:42 adapter_config.json
-rw-r--r-- 1 root root  17M Jun  9 04:42 adapter_model.safetensors
-rw-r--r-- 1 root root  907 Jun  9 04:43 all_results.json
-rw-r--r-- 1 root root 2.4K Jun  9 04:42 chat_template.jinja
drwxr-xr-x 2 root root 4.0K Jun  9 04:42 checkpoint-100/
drwxr-xr-x 2 root root 4.0K Jun  9 04:25 checkpoint-50/
-rw-r--r-- 1 root root  696 Jun  9 04:43 eval_results.json
-rw-r--r-- 1 root root 2.3K Jun  9 04:42 README.md
drwxr-xr-x 3 root root 4.0K Jun  9 04:09 runs/
-rw-r--r-- 1 root root  707 Jun  9 04:42 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  9 04:42 tokenizer.json
-rw-r--r-- 1 root root  71K Jun  9 04:42 trainer_state.json
-rw-r--r-- 1 root root 5.7K Jun  9 04:42 training_args.bin
-rw-r--r-- 1 root root  245 Jun  9 04:42 train_results.json


模型训练结果：
- 使用lora训练模型，则保存的lora权重是`adapter_model.safetensors`, lora配置文件是`adapter_config.json`，合并到base model的方法见`tools/merge_peft_adapter.py`
- 日志保存在`output_dir/runs`目录下，可以使用tensorboard查看，启动tensorboard方式如下：`tensorboard --logdir output_dir/runs --host 0.0.0.0 --port 8009`

lora模型权重合并到base model，合并后的模型保存在`--output_dir`目录下，合并方法如下：

In [49]:
!python tools/merge_peft_adapter.py \
    --base_model merged-sft --lora_model outputs-dpo-v1 --output_dir merged-dpo/

W0609 05:14:53.509000 35836 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0609 05:14:53.537000 35836 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Namespace(base_model='merged-sft', tokenizer_path=None, lora_model='outputs-dpo-v1', resize_emb=False, output_dir='merged-dpo/', hf_hub_model_id='', hf_hub_token=None)
Base model: merged-sft
LoRA model: outputs-dpo-v1
Loading LoRA for causal language model (archs=['Qwen2ForCausalLM'])
Loading weights: 100% 290/290 [00:00<00:00, 683.10it/s]
Merging with merge_and_unload...
Saving to Hugging Face format...
Writin

In [50]:
%ls -lh merged-dpo/

total 954M
-rw-r--r-- 1 root root 2.4K Jun  9 03:53 chat_template.jinja
-rw-r--r-- 1 root root 1.3K Jun  9 05:14 config.json
-rw-r--r-- 1 root root  139 Jun  9 03:53 generation_config.json
-rw-r--r-- 1 root root 943M Jun  9 05:15 model.safetensors
-rw-r--r-- 1 root root  697 Jun  9 03:53 tokenizer_config.json
-rw-r--r-- 1 root root  11M Jun  9 03:53 tokenizer.json


In [51]:
%cat merged-dpo/config.json

{
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 32768,
  "max_window_layers": 24,
  "model_type": "qwen2",
  "num_attention_heads": 14,
  "num_hidden_layers": 24,
  "num_key_value_heads": 2,
  "pad_token_id": n

Stage3 偏好建模第一次训练完成。

**至此一个完整的训练流程演示完成。**

# Test

In [52]:
!python demo/inference.py --base_model merged-dpo
# 或在shell中运行
# python demo/inference.py --base_model merged-dpo --interactive

W0609 05:18:07.041000 36677 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0609 05:18:07.068000 36677 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Namespace(base_model='merged-dpo', lora_model='', tokenizer_path=None, system_prompt='', stop_str='', repetition_penalty=1.0, max_new_tokens=512, data_file=None, interactive=False, single_tune=False, temperature=0.7, output_file='./predictions_result.jsonl', eval_batch_size=4, resize_emb=False, load_in_8bit=False, load_in_4bit=False)
Loading weights: 100% 290/290 [00:00<00:00, 830.78it/s]
Qwen2Tokenizer(name_or

Input:介绍下南京

Response:  南京市位于江苏省西南部，是全国首批历史文化名城、国家中心城市和自由贸易试验区。
